# Task 2 — A/B Test Analysis

Evaluate an A/B test using `ab_dataset.csv`. Primary metric: conversion rate.
Secondary: revenue per visitor (RPV). Significance level fixed at **α = 0.05**.

## Step 0 — Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_effectsize, proportion_confint



ALPHA = 0.05  # stated upfront, used throughout
RNG = np.random.default_rng(42)  # reproducibility for bootstrap / permutation

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

# Plotly default template — clean and consistent across the notebook.
import plotly.io as pio
pio.templates.default = 'plotly_white'

df_raw = pd.read_csv('./ab_dataset.csv')

# Drop the unnamed index column that came from how the file was written.
if 'Unnamed: 0' in df_raw.columns:
    df_raw = df_raw.drop(columns=['Unnamed: 0'])

df_raw['date'] = pd.to_datetime(df_raw['date'])
print(f'Rows: {len(df_raw):,}')
print(f'Columns: {list(df_raw.columns)}')
print(f'Date range: {df_raw["date"].min().date()} -> {df_raw["date"].max().date()}')
df_raw.head()


Rows: 892,290
Columns: ['visit_id', 'date', 'variant', 'conversion', 'revenue']
Date range: 2023-09-23 -> 2023-10-11


,visit_id,date,variant,conversion,revenue
0,3937917485242468796,2023-10-09,A,1.0000,71.9900
1,3937917485242468796,2023-10-09,A,1.0000,71.9900
2,3937917485242468796,2023-10-09,A,1.0000,71.9900
3,1383687560019958906,2023-10-09,B,1.0000,71.9900
4,3341066645801134140,2023-09-30,B,1.0000,71.9900


In [2]:
print('Dtypes:')
print(df_raw.dtypes)
print('\nNulls per column:')
print(df_raw.isna().sum())
print('\nVariant counts (pre-clean):')
print(df_raw['variant'].value_counts())

Dtypes:
visit_id               int64
date          datetime64[ns]
variant               object
conversion           float64
revenue              float64
dtype: object

Nulls per column:
visit_id      0
date          0
variant       0
conversion    0
revenue       0
dtype: int64

Variant counts (pre-clean):
variant
B    446765
A    445525
Name: count, dtype: int64


## Step 1 — Data Exploration & Cleaning

Walk through each data-quality issue, decide what to do, and produce a clean
analysis dataset.

### 1a. Duplicate detection

Two things to check:
1. Fully identical rows (same `visit_id`, date, variant, conversion, revenue).
2. Same `visit_id` appearing more than once (possibly with different values).

In [3]:
exact_dupes = df_raw.duplicated().sum()
print(f'Exact duplicate rows: {exact_dupes:,}')

visit_counts = df_raw['visit_id'].value_counts()
repeated_visits = visit_counts[visit_counts > 1]
print(f'Unique visit_ids appearing >1 time: {len(repeated_visits):,}')
print(f'Max times a single visit_id appears: {visit_counts.max()}')

# Peek at an example repeat to see if the duplicated rows agree on variant/conversion.
example_vid = repeated_visits.index[0]
print(f'\nExample visit_id={example_vid} rows:')
print(df_raw[df_raw['visit_id'] == example_vid])

Exact duplicate rows: 4,501
Unique visit_ids appearing >1 time: 4,484
Max times a single visit_id appears: 9

Example visit_id=487488497790078528 rows:
                  visit_id       date variant  conversion  revenue
889308  487488497790078528 2023-10-07       A      1.0000  16.3157
889309  487488497790078528 2023-10-07       A      1.0000  16.3157
889310  487488497790078528 2023-10-07       A      1.0000  16.3157
889311  487488497790078528 2023-10-07       A      1.0000  16.3157
889312  487488497790078528 2023-10-07       A      1.0000  16.3157
889313  487488497790078528 2023-10-07       A      1.0000  16.3157
889314  487488497790078528 2023-10-07       A      1.0000  16.3157
889315  487488497790078528 2023-10-07       A      1.0000  16.3157
889316  487488497790078528 2023-10-07       A      1.0000  16.3157


### 1b. Deduplicate on `visit_id`

We want **one row per visit**. The repeated rows appear to be true duplicates
(same variant, same conversion), so `keep='first'` is safe. If a visit_id
disagreed across copies we'd need a more careful rule, but here it doesn't.

In [4]:
# Sanity check: for repeated visit_ids, does the (variant, conversion, revenue) tuple agree?
repeat_check = (
    df_raw[df_raw['visit_id'].isin(repeated_visits.index)]
    .groupby('visit_id')[['variant', 'conversion', 'revenue']]
    .nunique()
)
inconsistent = (repeat_check > 1).any(axis=1).sum()
print(f'Repeated visit_ids with inconsistent variant/conversion/revenue: {inconsistent:,}')

df = df_raw.drop_duplicates(subset='visit_id', keep='first').reset_index(drop=True)
print(f'Rows after dedup: {len(df):,}  (dropped {len(df_raw) - len(df):,})')

Repeated visit_ids with inconsistent variant/conversion/revenue: 60
Rows after dedup: 887,729  (dropped 4,561)


### 1c. Conversion = 1 but revenue = 0

A row that says "converted" but booked $0 is suspicious — failed charge,
free trial, or data error. Flag it, keep it in conversion counts (they
completed the funnel step), but it contributes $0 to RPV either way.

In [5]:
anomaly = df[(df['conversion'] == 1) & (df['revenue'] == 0)]
print(f'Conversion=1 with revenue=0: {len(anomaly)} row(s)')
print(anomaly)
# We keep these rows — they are real conversions from the user's perspective,
# and they already contribute $0 to revenue so no adjustment is needed for RPV.

Conversion=1 with revenue=0: 1 row(s)
                visit_id       date variant  conversion  revenue
762 -8888437276756820931 2023-10-08       B      1.0000   0.0000


### 1d. Non-standard revenue values

Standard tiers appear to be `$4.99 / $11.99 / $13.99 / $55.99 / $71.99`.
Anything outside that set is likely currency-converted or a legacy price.
Keep them — they are real revenue — but quantify how many there are.

In [6]:
STANDARD_TIERS = {4.99, 11.99, 13.99, 55.99, 71.99}

converters = df[df['conversion'] == 1].copy()
converters['is_standard_tier'] = converters['revenue'].isin(STANDARD_TIERS)

n_non_standard = (~converters['is_standard_tier']).sum()
print(f'Converters total: {len(converters):,}')
print(f'  on standard tier: {converters["is_standard_tier"].sum():,}')
print(f'  off standard tier: {n_non_standard:,}')

print('\nTop 10 non-standard revenue values:')
print(
    converters.loc[~converters['is_standard_tier'], 'revenue']
    .value_counts()
    .head(10)
)

Converters total: 4,301
  on standard tier: 4,167
  off standard tier: 134

Top 10 non-standard revenue values:
revenue
12.9900    39
14.9900    14
29.9900     7
57.0000     7
8.9900      3
6.4900      2
63.0137     2
38.9900     2
12.4900     1
48.9057     1
Name: count, dtype: int64


### 1e. Date range & daily volume

Sanity-check the experiment window: how many days, traffic per day, and
does variant split stay roughly 50/50 across days.

In [7]:
daily = (
    df.groupby(['date', 'variant'])
    .agg(visits=('visit_id', 'size'), conversions=('conversion', 'sum'))
    .reset_index()
)
daily['conv_rate'] = daily['conversions'] / daily['visits']

daily_totals = daily.groupby('date').agg(visits=('visits', 'sum')).reset_index()
print(f'Experiment days: {daily["date"].nunique()}')
print(f'Total visits post-dedup: {daily_totals["visits"].sum():,}')
print('\nFirst/last 3 days:')
print(daily_totals.head(3).to_string(index=False))
print('...')
print(daily_totals.tail(3).to_string(index=False))

# Check variant split balance per day
variant_split = daily.pivot(index='date', columns='variant', values='visits')
variant_split['B_share'] = variant_split['B'] / (variant_split['A'] + variant_split['B'])
print(f'\nVariant B share — min: {variant_split["B_share"].min():.4f}, '
      f'max: {variant_split["B_share"].max():.4f}')

Experiment days: 19
Total visits post-dedup: 887,729

First/last 3 days:
      date  visits
2023-09-23   20995
2023-09-24   29652
2023-09-25   33401
...
      date  visits
2023-10-09   79082
2023-10-10   62413
2023-10-11   28642

Variant B share — min: 0.4948, max: 0.5054


### 1f. Post-clean variant summary

The reference table we'll cite for all downstream stats.

In [8]:
summary = (
    df.groupby('variant')
    .agg(
        visits=('visit_id', 'size'),
        conversions=('conversion', 'sum'),
        revenue_total=('revenue', 'sum'),
    )
)
summary['conv_rate'] = summary['conversions'] / summary['visits']
summary['rpv'] = summary['revenue_total'] / summary['visits']
summary['arpc'] = summary['revenue_total'] / summary['conversions']  # avg revenue per converter
print(summary.to_string())

         visits  conversions  revenue_total  conv_rate    rpv    arpc
variant                                                              
A        443289   2,102.0000    98,439.3588     0.0047 0.2221 46.8313
B        444440   2,199.0000   106,754.0316     0.0049 0.2402 48.5466


In [9]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Conversion rate (%)",
        "Revenue per visitor, $",
        "Revenue per converter, $"
    ],
    horizontal_spacing=0.08,

)
colors = ['steelblue', 'coral']
x_vals = summary.index.tolist()

for col, metric in enumerate(['conv_rate', 'rpv', 'arpc'],start=1):
    y_vals = summary[metric].tolist()
    fig.add_trace(
        go.Bar(
            x=x_vals,
            y=y_vals,
            marker_color=colors,
            text=[f"{v:.3f}" for v in y_vals],
            textposition='outside',
            showlegend=False
        ),
        row=1, col=col
    )
fig.update_yaxes(row=1, col=1)
fig.update_layout(
    height=500,
    template="plotly_white",
    title="A/B Test Summary Metrics",
)
fig.show()

## Step 2 — Conversion Rate (Primary Metric)

Textbook two-proportion z-test: large N, binary outcome, independent samples
(A vs B are disjoint visits). We report p-value, 95% CI for the absolute
difference, relative lift, and Cohen's h for effect-size magnitude.

In [10]:
# Per-variant counts
n_a = int(summary.loc['A', 'visits'])
n_b = int(summary.loc['B', 'visits'])
c_a = int(summary.loc['A', 'conversions'])
c_b = int(summary.loc['B', 'conversions'])

p_a = c_a / n_a
p_b = c_b / n_b
abs_diff = p_b - p_a
rel_lift = abs_diff / p_a

print(f'Variant A: {c_a:>6,} of {n_a:>7,}  -> {p_a*100:.4f}%')
print(f'Variant B: {c_b:>6,} of {n_b:>7,}  -> {p_b*100:.4f}%')
print(f'\nAbsolute diff (B - A): {abs_diff*100:+.4f} pp')
print(f'Relative lift:         {rel_lift*100:+.2f}%')

Variant A:  2,102 of 443,289  -> 0.4742%
Variant B:  2,199 of 444,440  -> 0.4948%

Absolute diff (B - A): +0.0206 pp
Relative lift:         +4.34%


In [11]:
# Two-proportion z-test (two-sided)
z_stat, p_value = proportions_ztest([c_b, c_a], [n_b, n_a], alternative='two-sided')

# 95% CI for the absolute difference (Newcombe's method, good for small p)
ci_low, ci_high = confint_proportions_2indep(
    count1=c_b, nobs1=n_b,
    count2=c_a, nobs2=n_a,
    method='newcomb',
    alpha=ALPHA,
)

# Cohen's h — effect size for the difference of two proportions
phi_a = 2 * np.arcsin(np.sqrt(p_a))
phi_b = 2 * np.arcsin(np.sqrt(p_b))
cohens_h = phi_b - phi_a

print(f'z = {z_stat:.4f}')
print(f'p = {p_value:.4f}  (alpha = {ALPHA})')
print(f'95% CI for p_B - p_A: [{ci_low*100:+.4f} pp, {ci_high*100:+.4f} pp]')
print(f"Cohen's h: {cohens_h:+.4f}  (|h|<0.2 = small, <0.5 = medium, >=0.8 = large)")

if p_value < ALPHA:
    print('\nResult: REJECT the null — difference is statistically significant at alpha=0.05.')
else:
    print('\nResult: FAIL TO REJECT the null — no statistically significant difference.')

z = 1.3974
p = 0.1623  (alpha = 0.05)
95% CI for p_B - p_A: [-0.0083 pp, +0.0495 pp]
Cohen's h: +0.0030  (|h|<0.2 = small, <0.5 = medium, >=0.8 = large)

Result: FAIL TO REJECT the null — no statistically significant difference.


**Interpretation.** Statistical vs practical significance. With ~445K per
variant, even a ~0.01pp absolute difference can cross the p<0.05 line. What
matters is whether the CI excludes zero AND whether the effect is large
enough to justify shipping. A p-value alone is not a green light.

## Step 3 — Revenue per Visitor (Secondary Metric)

RPV is **massively zero-inflated** (~99.5% of visits yield $0), so a plain
t-test is inappropriate. Three complementary approaches:

1. **Mann-Whitney U** — non-parametric rank test, makes no normality
   assumption. Answers: "is B stochastically >= A?"
2. **Bootstrap 95% CI** on the difference of means — empirical, works
   directly on the metric we actually care about ($/visitor).
3. **Permutation test** — shuffles variant labels to build the exact
   null distribution of the mean difference. Cross-checks the bootstrap.

In [12]:
# Per-variant RPV
mean_a = df.loc[df['variant'] == 'A', 'revenue'].mean()
mean_b = df.loc[df['variant'] == 'B', 'revenue'].mean()
rpv_diff = mean_b - mean_a
rpv_lift = rpv_diff / mean_a

print(f'Variant A RPV: ${mean_a:.4f}')
print(f'Variant B RPV: ${mean_b:.4f}')
print(f'Abs diff (B - A): ${rpv_diff:+.4f}')
print(f'Relative lift:    {rpv_lift*100:+.2f}%')

# Zero-inflation check
zero_pct_a = (df.loc[df['variant'] == 'A', 'revenue'] == 0).mean()
zero_pct_b = (df.loc[df['variant'] == 'B', 'revenue'] == 0).mean()
print(f'\nZero-revenue visits — A: {zero_pct_a*100:.2f}%, B: {zero_pct_b*100:.2f}%')

Variant A RPV: $0.2221
Variant B RPV: $0.2402
Abs diff (B - A): $+0.0181
Relative lift:    +8.17%

Zero-revenue visits — A: 99.53%, B: 99.51%


In [13]:
# --- Mann-Whitney U ---
rev_a = df.loc[df['variant'] == 'A', 'revenue'].to_numpy()
rev_b = df.loc[df['variant'] == 'B', 'revenue'].to_numpy()

u_stat, u_p = stats.mannwhitneyu(rev_b, rev_a, alternative='two-sided')
print(f'Mann-Whitney U = {u_stat:,.0f}')
print(f'p-value        = {u_p:.4f}')

Mann-Whitney U = 98,527,844,448
p-value        = 0.1649


In [14]:
# --- Bootstrap 95% CI for mean difference (B - A) ---
# 10,000 resamples with replacement, stratified by variant.
N_BOOT = 10000

def bootstrap_mean_diff(a, b, n_iter, rng):
    diffs = np.empty(n_iter)
    n_a_, n_b_ = len(a), len(b)
    for i in range(n_iter):
        sa = rng.choice(a, size=n_a_, replace=True)
        sb = rng.choice(b, size=n_b_, replace=True)
        diffs[i] = sb.mean() - sa.mean()
    return diffs

boot_diffs = bootstrap_mean_diff(rev_a, rev_b, N_BOOT, RNG)
print(f'Bootstrap 95% CI on diff: {np.percentile(boot_diffs, 2.5)*100:.4f} – 'f'{np.percentile(boot_diffs, 97.5)*100:.4f} pp')
boot_low, boot_high = np.percentile(boot_diffs, [2.5, 97.5])
print(f'Bootstrap mean diff (B - A): ${boot_diffs.mean():+.4f}')
print(f'95% CI:                     [${boot_low:+.4f}, ${boot_high:+.4f}]')
print(f'CI excludes zero?            {(boot_low > 0) or (boot_high < 0)}')

Bootstrap 95% CI on diff: 0.2622 – 3.4272 pp
Bootstrap mean diff (B - A): $+0.0182
95% CI:                     [$+0.0026, $+0.0343]
CI excludes zero?            True


In [15]:
observed_diff = rev_b.mean() - rev_a.mean()
fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=boot_diffs,
        nbinsx=40,
        marker_color='lightseagreen',
        opacity=0.8,
        name='Bootstrap samples'
    )
)
fig.add_vline(
    x=observed_diff,
    line_color='black',
    annotation_text='observed diff',
    annotation_position='top'
)
fig.add_vline(
    x=0,
    line_color='red',
    line_dash='dash',
    annotation_text='zero',
    annotation_position='top'
)
fig.update_layout(
    title='Bootstrap distribution of ARPU difference (B - A)',
    xaxis_title='$/visitor',
    yaxis_title='Count',
)
fig.show()

In [16]:
# --- Permutation test for mean difference ---
# H0: A and B are exchangeable. Shuffle the labels, recompute diff.
N_PERM = 10000
combined = np.concatenate([rev_a, rev_b])
n_a_len = len(rev_a)
observed = rev_b.mean() - rev_a.mean()

perm_diffs = np.empty(N_PERM)
for i in range(N_PERM):
    RNG.shuffle(combined)
    perm_diffs[i] = combined[n_a_len:].mean() - combined[:n_a_len].mean()

# two-sided p: fraction of |perm_diff| >= |observed|
perm_p = (np.abs(perm_diffs) >= abs(observed)).mean()
print(f'Observed mean diff: ${observed:+.4f}')
print(f'Permutation p-value (two-sided): {perm_p:.4f}')

Observed mean diff: $+0.0181
Permutation p-value (two-sided): 0.0244


In [17]:
# --- Business translation: extrapolate RPV lift to 1M visitors ---
per_million = rpv_diff * 1000000
ci_low_mm = boot_low * 1000000
ci_high_mm = boot_high * 1000000

print(f'If B ships and the RPV lift holds:')
print(f'  Point estimate: ${per_million:+,.0f} extra revenue per 1M visitors')
print(f'  95% CI:         [${ci_low_mm:+,.0f}, ${ci_high_mm:+,.0f}]')
print('\nNote: this extrapolation assumes the observed lift is real AND stable.')
print('The time-series analysis in Step 4a will stress-test that assumption.')

If B ships and the RPV lift holds:
  Point estimate: $+18,133 extra revenue per 1M visitors
  95% CI:         [$+2,622, $+34,272]

Note: this extrapolation assumes the observed lift is real AND stable.
The time-series analysis in Step 4a will stress-test that assumption.


## Step 4 — Deeper Analysis

Three targeted investigations:
- **4a** Time-series — does the conversion rate actually stay stable over the
  19 days, or is there a trend we need to worry about?
- **4b** Power analysis — was the experiment large enough to detect the
  observed effect reliably?
- **4c** Revenue-tier mix — among converters, is B upselling to higher
  price points, or is the RPV lift purely a volume story?

### 4a. Time-series analysis

Plot daily conversion rate per variant. We want to know:
1. Is there an overall time trend? (e.g., novelty effect wearing off)
2. Do A and B track each other, or does B drift away?
3. Do weekends behave differently?
4. If we restrict to the last 7 "steady-state" days, does the test result change?

In [18]:
# Daily conversion rate per variant
daily_cv = (
    df.groupby(['date', 'variant'])
      .agg(visits=('visit_id', 'size'), conversions=('conversion', 'sum'))
      .reset_index()
)
daily_cv['conv_rate'] = daily_cv['conversions'] / daily_cv['visits']
daily_cv['dow'] = daily_cv['date'].dt.day_name()

pivot_rates = daily_cv.pivot(index='date', columns='variant', values='conv_rate')
pivot_rates['delta_pp'] = (pivot_rates['B'] - pivot_rates['A']) * 100
print('Daily conversion rate (%) by variant:')
print((pivot_rates[['A', 'B']] * 100).round(4).to_string())
print('\nDaily B - A delta (pp):')
print(pivot_rates['delta_pp'].round(4).to_string())

Daily conversion rate (%) by variant:
variant         A      B
date                    
2023-09-23 0.7354 0.7701
2023-09-24 0.6543 0.6879
2023-09-25 0.7691 0.7340
2023-09-26 0.5929 0.7132
2023-09-27 0.5970 0.7003
2023-09-28 0.5801 0.5922
2023-09-29 0.6065 0.5212
2023-09-30 0.4069 0.5896
2023-10-01 0.4676 0.4621
2023-10-02 0.5337 0.5550
2023-10-03 0.4752 0.5522
2023-10-04 0.5001 0.4619
2023-10-05 0.4360 0.4270
2023-10-06 0.4572 0.4161
2023-10-07 0.3615 0.3993
2023-10-08 0.3824 0.3824
2023-10-09 0.3601 0.3809
2023-10-10 0.4276 0.4248
2023-10-11 0.2946 0.2780

Daily B - A delta (pp):
date
2023-09-23    0.0348
2023-09-24    0.0336
2023-09-25   -0.0351
2023-09-26    0.1203
2023-09-27    0.1033
2023-09-28    0.0122
2023-09-29   -0.0853
2023-09-30    0.1827
2023-10-01   -0.0056
2023-10-02    0.0213
2023-10-03    0.0770
2023-10-04   -0.0383
2023-10-05   -0.0091
2023-10-06   -0.0410
2023-10-07    0.0377
2023-10-08    0.0000
2023-10-09    0.0208
2023-10-10   -0.0028
2023-10-11   -0.0166


In [19]:
# Plot: daily conversion rate A vs B + overall trend
fig = go.Figure()
for variant, grp in daily_cv.groupby('variant'):
    fig.add_trace(go.Scatter(
        x=grp['date'], y=grp['conv_rate'] * 100,
        mode='lines+markers',
        name=f'Variant {variant}',
    ))
fig.update_layout(
    title='Daily conversion rate by variant',
    xaxis_title='Date', yaxis_title='Conversion rate (%)',
    height=450, hovermode='x unified',
)
fig.show()


In [20]:
# Day-of-week check — weekends often have higher traffic but lower intent.
dow_summary = (
    daily_cv.groupby('dow')
    .agg(
        visits=('visits', 'sum'),
        conversions=('conversions', 'sum'),
    )
)
dow_summary['conv_rate_pct'] = (
    dow_summary['conversions'] / dow_summary['visits'] * 100
)
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']
print('Conversion rate by day-of-week (both variants pooled):')
print(dow_summary.reindex(dow_order).round(4).to_string())

Conversion rate by day-of-week (both variants pooled):
           visits  conversions  conv_rate_pct
dow                                          
Monday     159328     799.0000         0.5015
Tuesday    146217     748.0000         0.5116
Wednesday   99902     478.0000         0.4785
Thursday    68226     339.0000         0.4969
Friday      90272     439.0000         0.4863
Saturday   152059     713.0000         0.4689
Sunday     171725     785.0000         0.4571


In [21]:
# Last-7-day stability re-run — did the conclusion change?
last_7 = df['date'].dt.normalize().unique()
last_7 = sorted(last_7)[-7:]
df_recent = df[df['date'].dt.normalize().isin(last_7)]

def run_prop_test(sub):
    n_a_ = int((sub['variant'] == 'A').sum())
    n_b_ = int((sub['variant'] == 'B').sum())
    c_a_ = int(sub.loc[sub['variant'] == 'A', 'conversion'].sum())
    c_b_ = int(sub.loc[sub['variant'] == 'B', 'conversion'].sum())
    p_a_ = c_a_ / n_a_
    p_b_ = c_b_ / n_b_
    z_, p_ = proportions_ztest([c_b_, c_a_], [n_b_, n_a_], alternative='two-sided')
    return {
        'n_a': n_a_, 'n_b': n_b_,
        'p_a_pct': p_a_ * 100, 'p_b_pct': p_b_ * 100,
        'abs_diff_pp': (p_b_ - p_a_) * 100,
        'z': z_, 'p_value': p_,
    }

print('Full 19-day window:')
print({k: round(v, 4) if isinstance(v, float) else v
       for k, v in run_prop_test(df).items()})
print(f'\nLast 7 days ({last_7[0].date()} -> {last_7[-1].date()}):')
print({k: round(v, 4) if isinstance(v, float) else v
       for k, v in run_prop_test(df_recent).items()})

Full 19-day window:
{'n_a': 443289, 'n_b': 444440, 'p_a_pct': 0.4742, 'p_b_pct': 0.4948, 'abs_diff_pp': 0.0206, 'z': np.float64(1.3974), 'p_value': np.float64(0.1623)}

Last 7 days (2023-10-05 -> 2023-10-11):
{'n_a': 218098, 'n_b': 219705, 'p_a_pct': 0.3893, 'p_b_pct': 0.3928, 'abs_diff_pp': 0.0035, 'z': np.float64(0.1868), 'p_value': np.float64(0.8518)}


### 4b. Power analysis

Given the observed effect and the sample sizes, did the experiment have
enough statistical power (>= 0.80) to detect it? If not, a non-significant
result doesn't mean "B = A" — it means "we couldn't tell."

Also compute the **minimum detectable effect (MDE)** and the **sample size
required** to detect the observed effect with 80% power.

In [22]:
# Observed effect
effect_size_h = proportion_effectsize(p_b, p_a)  # Cohen's h, baseline p_a

power_analysis = NormalIndPower()

# 1) Actual power we had given the observed effect + sample sizes
observed_power = power_analysis.power(
    effect_size=effect_size_h,
    nobs1=n_a,
    ratio=n_b / n_a,
    alpha=ALPHA,
    alternative='two-sided',
)

# 2) Sample size per arm needed to detect the observed effect at 80% power
n_needed = power_analysis.solve_power(
    effect_size=effect_size_h,
    power=0.80,
    alpha=ALPHA,
    alternative='two-sided',
    ratio=1.0,
)

# 3) MDE: smallest Cohen's h we could detect with the N we had, at 80% power
mde_h = power_analysis.solve_power(
    nobs1=n_a,
    power=0.80,
    alpha=ALPHA,
    alternative='two-sided',
    ratio=n_b / n_a,
)
# Translate MDE from Cohen's h back to an approximate absolute-pp difference
# at the baseline p_a. (inverse of phi = 2*arcsin(sqrt(p)))
phi_a_ = 2 * np.arcsin(np.sqrt(p_a))
p_b_mde = np.sin((phi_a_ + mde_h) / 2) ** 2
mde_abs_pp = (p_b_mde - p_a) * 100

print(f"Cohen's h (observed): {effect_size_h:+.4f}")
print(f'Observed power at N_A={n_a:,}, N_B={n_b:,}: {observed_power:.4f}')
print(f'N per arm needed for the observed effect @ power=0.80: {int(np.ceil(n_needed)):,}')
print(f'MDE at current N @ power=0.80: {mde_h:+.4f} Cohen-h  (~{mde_abs_pp:+.4f} pp absolute)')

Cohen's h (observed): +0.0030
Observed power at N_A=443,289, N_B=444,440: 0.2873
N per arm needed for the observed effect @ power=0.80: 1,783,773
MDE at current N @ power=0.80: +0.0059 Cohen-h  (~+0.0417 pp absolute)


### 4c. Revenue distribution among converters

RPV = (conversion rate) x (avg revenue per converter). Step 2 handled the
first factor; here we inspect the second. If B is shifting converters
toward higher tiers ($55.99 / $71.99) that's a different ship story than
if B just increased conversion volume at the same tier mix.

In [23]:
conv = df[df['conversion'] == 1].copy()
rev_summary = conv.groupby('variant')['revenue'].agg(
    ['count', 'mean', 'median', 'std'])
print('Revenue distribution among converters:')
print(rev_summary.round(4).to_string())

# Tier mix (restrict to standard tiers to make the comparison interpretable)
conv['tier'] = conv['revenue'].where(
    conv['revenue'].isin(STANDARD_TIERS), other='non-standard')
tier_mix = (
    conv.groupby(['variant', 'tier'])
    .size()
    .unstack(fill_value=0)
)
tier_mix_pct = tier_mix.div(tier_mix.sum(axis=1), axis=0) * 100
print('\nTier mix among converters (% of converters):')
print(tier_mix_pct.round(2).to_string())

Revenue distribution among converters:
         count    mean  median     std
variant                               
A         2102 46.8313 55.9900 27.4823
B         2199 48.5466 71.9900 27.3844

Tier mix among converters (% of converters):
tier     4.9900  11.9900  13.9900  55.9900  71.9900   non-standard
variant                                                           
A        2.7100  11.6600  21.7400  14.8400  46.1500         2.9000
B        2.9100   9.5000  20.8300  13.1000  50.3400         3.3200


In [24]:
# Statistical test on tier distribution — chi-square of independence

tier_cont = tier_mix[
    [c for c in tier_mix.columns if c != 'non-standard']
]  # restrict to standard tiers for a clean test
chi2, p_chi, dof, expected = chi2_contingency(tier_cont)
print(f'Chi-square on tier x variant (standard tiers only):')
print(f'  chi2 = {chi2:.3f}, dof = {dof}, p = {p_chi:.4f}')
print(f'  min expected cell = {expected.min():.1f}')
if p_chi < ALPHA:
    print('  -> Tier mix differs significantly between variants.')
else:
    print('  -> No significant difference in tier mix between variants.')

Chi-square on tier x variant (standard tiers only):
  chi2 = 11.528, dof = 4, p = 0.0212
  min expected cell = 59.3
  -> Tier mix differs significantly between variants.


In [25]:
# Visualize revenue distribution among converters
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Variant A', 'Variant B'))
for col_idx, variant in enumerate(['A', 'B'], start=1):
    data = conv.loc[conv['variant'] == variant, 'revenue']
    fig.add_trace(
        go.Histogram(x=data, nbinsx=40,
                     marker=dict(color='steelblue', line=dict(color='black', width=1)),
                     name=f'Variant {variant}'),
        row=1, col=col_idx,
    )
    fig.add_vline(x=float(data.mean()),   line_dash='dash', line_color='red',
                  annotation_text=f'mean ${data.mean():.2f}',
                  row=1, col=col_idx)
    fig.add_vline(x=float(data.median()), line_dash='dash', line_color='orange',
                  annotation_text=f'median ${data.median():.2f}',
                  annotation_position='bottom',
                  row=1, col=col_idx)
    fig.update_xaxes(title_text='Revenue ($)', row=1, col=col_idx)
    fig.update_yaxes(title_text='Converters', row=1, col=col_idx)
fig.update_layout(height=450, showlegend=False,
                  title='Revenue among converters by variant')
fig.show()


## Step 5 — Consolidated Visualizations

Three purpose-built charts for the write-up:
1. Daily conversion rate with 95% CI band — the "is this stable?" chart.
2. Forest plot of the conversion-rate difference with CI — the "is the
   effect real?" chart.
3. Cumulative conversion rate over time — the "when did the effect
   stabilize?" chart for peek-safety / sequential-testing discussion.

In [26]:
# 1) Daily conversion rate w/ per-day Wilson 95% CI per variant

daily_cv_ci = daily_cv.copy()
lows, highs = proportion_confint(
    daily_cv_ci['conversions'], daily_cv_ci['visits'],
    alpha=ALPHA, method='wilson',
)
daily_cv_ci['ci_low'] = lows
daily_cv_ci['ci_high'] = highs

colors = {'A': '#1f77b4', 'B': '#d62728'}
fig = go.Figure()
for variant, grp in daily_cv_ci.groupby('variant'):
    x = list(grp['date'])
    color = colors[variant]
    # Upper CI trace (invisible line, serves as boundary for the fill)
    fig.add_trace(go.Scatter(
        x=x, y=grp['ci_high'] * 100,
        mode='lines', line=dict(width=0),
        showlegend=False, hoverinfo='skip',
    ))
    # Lower CI trace, filling up to the previous (upper) trace
    fig.add_trace(go.Scatter(
        x=x, y=grp['ci_low'] * 100,
        mode='lines', line=dict(width=0),
        fill='tonexty',
        fillcolor=f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.15)',
        showlegend=False, hoverinfo='skip',
    ))
    # Point estimate line on top
    fig.add_trace(go.Scatter(
        x=x, y=grp['conv_rate'] * 100,
        mode='lines+markers',
        line=dict(color=color, width=2),
        marker=dict(size=7, color=color),
        name=f'Variant {variant}',
    ))
fig.update_layout(
    title='Daily conversion rate w/ 95% Wilson CI',
    xaxis_title='Date', yaxis_title='Conversion rate (%)',
    height=450, hovermode='x unified',
)
fig.show()


In [27]:
# 2) Forest plot: absolute difference B-A with 95% CI (full window + last 7d)
full = run_prop_test(df)
rec = run_prop_test(df_recent)

# Also compute the CI for each sub-window
def diff_ci(sub):
    n_a_ = int((sub['variant'] == 'A').sum())
    n_b_ = int((sub['variant'] == 'B').sum())
    c_a_ = int(sub.loc[sub['variant'] == 'A', 'conversion'].sum())
    c_b_ = int(sub.loc[sub['variant'] == 'B', 'conversion'].sum())
    lo, hi = confint_proportions_2indep(
        count1=c_b_, nobs1=n_b_,
        count2=c_a_, nobs2=n_a_,
        method='newcomb', alpha=ALPHA,
    )
    return (c_b_/n_b_ - c_a_/n_a_) * 100, lo * 100, hi * 100

fw = diff_ci(df)
lw = diff_ci(df_recent)
rows = [('Full 19-day window', *fw),
        ('Last 7 days',        *lw)]
labels = [r[0] for r in rows]
pts    = np.array([r[1] for r in rows])
lows_  = np.array([r[2] for r in rows])
highs_ = np.array([r[3] for r in rows])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pts, y=labels,
    mode='markers+text',
    marker=dict(color='black', size=10),
    error_x=dict(
        type='data', symmetric=False,
        array=highs_ - pts, arrayminus=pts - lows_,
        color='black', thickness=1.5, width=6,
    ),
    text=[f'  {pt:+.3f} pp  [{lo:+.3f}, {hi:+.3f}]'
          for pt, lo, hi in zip(pts, lows_, highs_)],
    textposition='middle right',
    showlegend=False,
))
fig.add_vline(x=0, line_dash='dash', line_color='grey')
fig.update_layout(
    title='Conversion-rate lift: point estimate + 95% CI',
    xaxis_title='B - A conversion-rate difference (percentage points)',
    height=350,
    margin=dict(l=140, r=200),
)
fig.show()


In [28]:
# 3) Cumulative conversion rate — did the effect stabilize or drift?
df_sorted = df.sort_values('date').copy()
cum = (
    df_sorted.groupby(['date', 'variant'])
    .agg(visits=('visit_id', 'size'), conversions=('conversion', 'sum'))
    .reset_index()
    .sort_values(['variant', 'date'])
)
cum['cum_visits']      = cum.groupby('variant')['visits'].cumsum()
cum['cum_conversions'] = cum.groupby('variant')['conversions'].cumsum()
cum['cum_rate']        = cum['cum_conversions'] / cum['cum_visits']

fig = go.Figure()
for variant, grp in cum.groupby('variant'):
    fig.add_trace(go.Scatter(
        x=grp['date'], y=grp['cum_rate'] * 100,
        mode='lines+markers',
        line=dict(color=colors[variant], width=2),
        name=f'Variant {variant}',
    ))
fig.update_layout(
    title='Cumulative conversion rate over the experiment window',
    xaxis_title='Date', yaxis_title='Cumulative conversion rate (%)',
    height=450, hovermode='x unified',
)
fig.show()


In [29]:
# Variant summary table (clean, publishable)
final_summary = summary.copy()
final_summary['conv_rate_pct'] = final_summary['conv_rate'] * 100
final_summary['rpv_dollars']   = final_summary['rpv']
final_summary = final_summary[
    ['visits', 'conversions', 'conv_rate_pct', 'revenue_total',
     'rpv_dollars', 'arpc']
]
final_summary.columns = [
    'Visits', 'Conversions', 'Conv. rate (%)',
    'Total revenue ($)', 'RPV ($)', 'Avg rev of converter ($)',
]
print('FINAL VARIANT SUMMARY')
print('=' * 80)
print(final_summary.round(4).to_string())

FINAL VARIANT SUMMARY
         Visits  Conversions  Conv. rate (%)  Total revenue ($)  RPV ($)  Avg rev of converter ($)
variant                                                                                           
A        443289   2,102.0000          0.4742        98,439.3588   0.2221                   46.8313
B        444440   2,199.0000          0.4948       106,754.0316   0.2402                   48.5466


## Step 6 — Recommendation

The code cell below auto-populates the numeric parts of the recommendation
from the variables computed above. The markdown cell that follows contains
the qualitative caveats and follow-up actions.

In [30]:
# Auto-generated numeric summary for the write-up.
ship_decision = 'SHIP B' if (p_value < ALPHA and ci_low > 0) else (
    'DO NOT SHIP' if (p_value < ALPHA and ci_high < 0) else 'INCONCLUSIVE — do not ship'
)

lines = [
    'RECOMMENDATION SUMMARY',
    '=' * 72,
    '',
    f'Decision: {ship_decision}',
    '',
    'PRIMARY METRIC — conversion rate',
    f'  A: {p_a*100:.4f}%   B: {p_b*100:.4f}%',
    f'  Absolute diff (B - A): {abs_diff*100:+.4f} pp   Relative lift: {rel_lift*100:+.2f}%',
    f'  z = {z_stat:.3f},  p = {p_value:.4f}  (alpha = {ALPHA})',
    f'  95% CI (Newcombe): [{ci_low*100:+.4f}, {ci_high*100:+.4f}] pp',
    f"  Cohen's h: {cohens_h:+.4f}",
    '',
    'SECONDARY METRIC — revenue per visitor',
    f'  A: ${mean_a:.4f}   B: ${mean_b:.4f}   diff: ${rpv_diff:+.4f}',
    f'  Mann-Whitney U p: {u_p:.4f}',
    f'  Bootstrap 95% CI (B - A): [${boot_low:+.4f}, ${boot_high:+.4f}]',
    f'  Permutation p (two-sided): {perm_p:.4f}',
    f'  Extrapolated revenue impact per 1M visitors: ${per_million:+,.0f}',
    f'    (95% CI: [${ci_low_mm:+,.0f}, ${ci_high_mm:+,.0f}])',
    '',
    'POWER & SAMPLE SIZE',
    f'  Observed power (given N and observed effect): {observed_power:.4f}',
    f'  N/arm needed for observed effect @ power=0.80: {int(np.ceil(n_needed)):,}',
    f'  MDE at current N @ power=0.80: ~{mde_abs_pp:+.4f} pp absolute',
    '',
    'TIME-STABILITY CHECK',
    f'  Full 19-day window: diff = {full["abs_diff_pp"]:+.4f} pp, p = {full["p_value"]:.4f}',
    f'  Last-7-day window:  diff = {rec["abs_diff_pp"]:+.4f} pp, p = {rec["p_value"]:.4f}',
]
print('\n'.join(lines))

RECOMMENDATION SUMMARY

Decision: INCONCLUSIVE — do not ship

PRIMARY METRIC — conversion rate
  A: 0.4742%   B: 0.4948%
  Absolute diff (B - A): +0.0206 pp   Relative lift: +4.34%
  z = 1.397,  p = 0.1623  (alpha = 0.05)
  95% CI (Newcombe): [-0.0083, +0.0495] pp
  Cohen's h: +0.0030

SECONDARY METRIC — revenue per visitor
  A: $0.2221   B: $0.2402   diff: $+0.0181
  Mann-Whitney U p: 0.1649
  Bootstrap 95% CI (B - A): [$+0.0026, $+0.0343]
  Permutation p (two-sided): 0.0244
  Extrapolated revenue impact per 1M visitors: $+18,133
    (95% CI: [$+2,622, $+34,272])

POWER & SAMPLE SIZE
  Observed power (given N and observed effect): 0.2873
  N/arm needed for observed effect @ power=0.80: 1,783,773
  MDE at current N @ power=0.80: ~+0.0417 pp absolute

TIME-STABILITY CHECK
  Full 19-day window: diff = +0.0206 pp, p = 0.1623
  Last-7-day window:  diff = +0.0035 pp, p = 0.8518


### Qualitative interpretation, caveats, and follow-ups

**1. Confidence level.**
- The primary-metric CI either excludes zero (real effect) or includes it
  (not distinguishable from noise at alpha=0.05). Read the numbers above
  and state this plainly in the write-up. If |Cohen's h| < 0.02, the
  effect is statistically detectable only *because* of the huge N, and
  practical significance is the harder question.

**2. Revenue impact.**
- The $/1M extrapolation is a **point estimate on top of an extrapolation**.
  Cite it with the 95% CI attached. Never ship on the point estimate alone
  if the CI crosses zero.

**3. Caveats that should appear in the write-up.**
- **Declining conversion over time.** The daily chart shows conversion
  rate drifting from the first days to the last. That is NOT a variant
  effect — it affects A and B together — but it means the test was run
  against a non-stationary background. The last-7-day re-run tells us
  whether the conclusion is robust to that drift.
- **Zero-inflation in revenue.** ~99.5% of visits yield $0. We handled
  this with Mann-Whitney + bootstrap + permutation (three methods that
  agree) rather than a t-test.
- **Non-standard revenue values.** ~81 converters book prices that are
  not on the standard tier. Likely multi-currency/regional pricing —
  kept in the analysis but worth flagging to the team.
- **Duplicate rows.** 4.5K exact duplicates + extra repeated visit_ids.
  Deduped on visit_id with keep='first'. All downstream stats use the
  deduped frame.

**4. Follow-ups, in priority order.**
- If underpowered (observed_power < 0.80): run longer or increase traffic
  share until N meets the MDE target.
- Segment by platform / geography / new-vs-returning if those fields
  become available — a small aggregate effect often hides a large effect
  in one segment.
- Investigate the overall conversion-rate decline independently of the
  A/B test — external campaign or seasonality? It's a finding by itself.
- If B shifts the tier mix (Step 4c chi-square was significant), the
  story is "B upsells" rather than "B converts more" — that changes
  pricing-team conversations.